# CFU prediction using TPM values from bulk RNA-seq

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import numpy as np
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir /  "cfus")

else:
  fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"
  cfu_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/cfus"

Load TPM data and CFU counts.

In [ ]:
from src.tpm_data import get_all_tpm_data

# Get data
data_df = get_all_tpm_data(
    fcnts_path = fcnts_path,
    cfu_path = cfu_path
)

# Find idx where CFU = 0, then list the sample ID
zero_idx = np.where(data_df["CFU"] == 0)[0]
print(data_df.index[zero_idx].tolist())

# Check CFU values for the other 2 replicates
rep_names = ["34CEF4hr-a", "34CEF4hr-b"]
cfu_val_check = data_df.loc[rep_names]["CFU"].tolist()
print(f"CFU values for 34CEF4hr-a and 34CEF4hr-b :{cfu_val_check}")

# Remove sample and convert to log 10 CFU
data_df = data_df[data_df["CFU"] != 0]
data_df["CFU"] = np.log10(data_df["CFU"])

## Training with stratified split

Nested CV for PLS regression with split stratified by drug.

In [ ]:
from src.split import combination_stratified_split
from src.train import run_nested_pls_cv

# Make stratified splits
strat_splits = combination_stratified_split(data_df, num_folds = 5, seed = 111)

# Run Nested CV
scores, mean_score = run_nested_pls_cv(
    df = data_df,
    splits = strat_splits,
    synergy = False
)

print(f"R^2 for 5 folds : {scores}")
print(f"Mean R^2 : {mean_score}")

## Training with sparse combination matrix

Train on only single-drug data as baseline.

In [ ]:
from src.train import train_custom_cfu_model

single_mask = data_df["num_drugs"] == 1
combo_mask = data_df["num_drugs"] == 2

forward_model = train_custom_cfu_model(
    df = data_df,
    train_mask = single_mask,
    test_mask = combo_mask,
    title = "Results for PLS regression model trained only on single-drug combination data"
)

Train on single-drug data + forward diagonal of combination matrix.

In [ ]:
forward_mask = (data_df["num_drugs"] == 1) | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))
forward_mask_complement = ~forward_mask

forward_model = train_custom_cfu_model(
    df = data_df,
    train_mask = forward_mask,
    test_mask = forward_mask_complement,
    title = "Results for PLS regression model trained on single-drug and diagonal combination data"
)

Train on single-drug data and dose-swapped diagonal of combination matrix.

In [ ]:
# Define mapping to cross doses
cross_dose_map = {
    0.25: 0.75,
    0.33: 0.50,
    0.50: 0.33,
    0.75: 0.25
}

# Map drug 1 dose to corresponding drug 2 dose, then generate boolean mask
mapped = np.array([cross_dose_map.get(dose, np.nan) for dose in data_df["drug1_dose"].to_numpy()])
backward_mask = data_df["drug2_dose"].to_numpy() == mapped
backward_mask = (data_df["num_drugs"] == 1) | backward_mask
backward_mask_complement = ~backward_mask

forward_model = train_custom_cfu_model(
    df = data_df,
    train_mask = backward_mask,
    test_mask = backward_mask_complement,
    title = "Results for PLS regression model trained on single-drug and backward diagonal combination data"
)

## Training with successively more combination data

Implement a gradient of increasing combination data. Add 10 random combination datapoints along x-axis. Each point along the x-axis should contain X different splits, allowing for consistent evaluation. Then, show the mean and std of the R^2 on the y-axis for each datapoint. Look for a plateau of R^2. Then isolate that point and look at mechanistic underpinnings. (or look at importances over time??)

In [ ]:
from src.train import plot_r2_over_data_increase

plot_r2_over_data_increase(
    df = data_df,
    step_size = 36,
    n_splits = 20,
    seed = 111
)

## Optimizing necessary combinations (fixed test set)

Fixed test set mask.

In [ ]:
# Define fixed test set mask
mask1 = (data_df["drug1_dose"] == 0.50) & (data_df["drug2_dose"] == 0.33)
mask2 = (data_df["drug1_dose"] == 0.75) & (data_df["drug2_dose"] == 0.33)
mask3 = (data_df["drug1_dose"] == 0.75) & (data_df["drug2_dose"] == 0.50)
mask4 = (data_df["drug1_dose"] == 0.33) & (data_df["drug2_dose"] == 0.50)
mask5 = (data_df["drug1_dose"] == 0.33) & (data_df["drug2_dose"] == 0.75) 
mask6 = (data_df["drug1_dose"] == 0.50) & (data_df["drug2_dose"] == 0.75) 

test_mask = mask1 | mask2 | mask3 | mask4 | mask5 | mask6

Train on diagonal.

In [ ]:
train_custom_cfu_model(
    df = data_df,
    train_mask = forward_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and diagonal combination data"
)

Train on first row.

In [ ]:
row1_mask = (data_df["num_drugs"] == 1) | (data_df["drug1_dose"] == 0.25)

train_custom_cfu_model(
    df = data_df,
    train_mask = row1_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and row 1 combination data"
)

Train on first column.

In [ ]:
col1_mask = (data_df["num_drugs"] == 1) | (data_df["drug2_dose"] == 0.25)

train_custom_cfu_model(
    df = data_df,
    train_mask = col1_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and column 1 combination data"
)

Train on first column and row.

In [ ]:
t_mask = (data_df["num_drugs"] == 1) | (data_df["drug1_dose"] == 0.25) | (data_df["drug2_dose"] == 0.25)

train_custom_cfu_model(
    df = data_df,
    train_mask = t_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and row 1 + column 1 combination data"
)

Train on first row and diagonal.

In [ ]:
row1_diag_mask = (data_df["num_drugs"] == 1) | (data_df["drug1_dose"] == 0.25) | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_cfu_model(
    df = data_df,
    train_mask = row1_diag_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and row 1 + diagonal combination data"
)

Train on first column and diagonal.

In [ ]:
col1_diag_mask = (data_df["num_drugs"] == 1) | (data_df["drug2_dose"] == 0.25) | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_cfu_model(
    df = data_df,
    train_mask = col1_diag_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and column 1 + diagonal combination data"
)

Train on first row, first column, and diagonal.

In [ ]:
all_mask = (data_df["num_drugs"] == 1) | (data_df["drug1_dose"] == 0.25)| (data_df["drug2_dose"] == 0.25) | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_cfu_model(
    df = data_df,
    train_mask = all_mask,
    test_mask = test_mask,
    title = "Results for PLS regression model trained on single-drug and row 1 + column 1 + diagonal combination data"
)